# RAG
> #### Just a PoC for the time being.

## Setup

### Importing the necessary libraries

In [19]:
# Base utils.
from os import getenv
from dotenv import load_dotenv

from IPython.display import Markdown
import warnings

# Model init. and invocation
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.tools import tool

# RAG
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

### Supressing warnings

In [ ]:
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Specific LangChain / LangGraph / Pydantic noise
warnings.filterwarnings("ignore", module="langchain")
warnings.filterwarnings("ignore", module="langgraph")
warnings.filterwarnings("ignore", module="pydantic")

True

### Environment config.

In [ ]:
load_dotenv()

OLLAMA_MODEL = getenv("OLLAMA_MODEL")
OLLAMA_EMBEDDING_MODEL = getenv("OLLAMA_EMBEDDING_MODEL")

## Semantic search

### Splitting & chunking

In [23]:
pdf_filepath = r"resources/acmecorp-employee-handbook.pdf"
loader = PyPDFLoader(pdf_filepath)

data = loader.load()
print(type(data))

incorrect startxref pointer(1)
parsing for Object Streams


<class 'list'>


In [25]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(f"Number of splits: {len(all_splits)}")

Number of splits: 3


### Vectorization

In [26]:
embeddings = OllamaEmbeddings(model=OLLAMA_EMBEDDING_MODEL)  # type: ignore
vector_store = InMemoryVectorStore(embeddings)

ids = vector_store.add_documents(documents=all_splits)

In [29]:
query = "What is the NDA policy for this organization?"
results = vector_store.similarity_search(query=query)

print(results[0])

page_content='Employee Handbook
Non-Disclosure Agreement (NDA) Policy
Employees must protect confidential information belonging to the company, its clients, and partners.
This includes, but is not limited to, product roadmaps, customer data, internal communications,
proprietary algorithms, financial information, and unreleased features. Confidential information may not
be shared with unauthorized individuals inside or outside the organization. These obligations continue
after employment ends.
Workplace Conduct Policy
Employees must maintain a respectful, professional environment free from harassment, discrimination,
and intimidation. All employees are expected to follow organizational values, collaborate effectively,
and communicate constructively. Disruptive behavior, verbal abuse, or misuse of company systems is
prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of servic

## RAG Agent

In [30]:
@tool
def search_handbook(query: str) -> str:
    """Sift through the employee handbook for relevant information."""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [31]:
SYS_PROMPT = (
    "You are a helpful agent that can search the employee handbook for information."
)

agent = create_agent(
    model=OLLAMA_MODEL,  
    tools=[search_handbook],
    system_prompt=SYS_PROMPT,
)

In [32]:
USR_PROMPT = "How many days of vacation does an employee get in their first year?"
response = agent.invoke({"messages": [HumanMessage(content=USR_PROMPT)]})

In [33]:
Markdown(response["messages"][-1].content)

Full-time employees receive **10 days** of Paid Time Off (PTO) per year during their first year of service (accruing at a rate of 0.833 days per month).